<a href="https://colab.research.google.com/github/VictorHugoTesti/FATEC-Processamento_Linguagem_Natural/blob/main/%5BPLN%5D_Aula10_1014.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Instalação e download de recursos

In [1]:
!pip install -q spacy scikit-learn pandas
!python -m spacy download pt_core_news_sm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.0/13.0 MB 78.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pt_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import random
import spacy
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.pipeline import make_pipeline

In [3]:
nlp = spacy.load("pt_core_news_sm")
print("Ambiente configurado com sucesso.")

Ambiente configurado com sucesso.


# Pipeline 1

In [4]:
aspectos = ["O atendimento", "O corretor", "A visita", "A negociação", "O apartamento",
            "O condomínio", "A entrega das chaves", "O contrato", "O financiamento", "A vistoria"]

avaliacoes_positivas = ["foi excelente", "foi rápido e eficiente", "superou minhas expectativas",
                        "foi ótimo, recomendo", "foi tranquilo do início ao fim",
                        "me deixou muito satisfeito", "foi muito bem feito",
                        "foi claro e organizado", "foi perfeito", "merece nota dez"]

avaliacoes_negativas = ["foi péssimo", "atrasou demais", "me deixou decepcionado",
                        "foi ruim, não recomendo", "foi confuso e demorado", "não foi resolvido",
                        "deu muito problema", "foi desorganizado", "foi uma decepção",
                        "não atendeu ao combinado"]

random.seed(42)

In [5]:
positivos = random.sample([f"{a} {p}." for a in aspectos for p in avaliacoes_positivas], 50)
negativos = random.sample([f"{a} {n}." for a in aspectos for n in avaliacoes_negativas], 50)

textos = positivos + negativos
rotulos = ["Positivo"] * 50 + ["Negativo"] * 50

print(f"Total de avaliações: {len(textos)}")
for t, r in list(zip(textos, rotulos))[:3] + list(zip(textos, rotulos))[50:53]:
    print(f"{r:9} | {t}")

Total de avaliações: 100
Positivo  | O financiamento foi rápido e eficiente.
Positivo  | O corretor foi tranquilo do início ao fim.
Positivo  | O atendimento foi ótimo, recomendo.
Negativo  | A entrega das chaves foi uma decepção.
Negativo  | O corretor não foi resolvido.
Negativo  | O apartamento foi uma decepção.


In [6]:
PALAVRAS_NEGACAO = {"não", "nunca", "jamais", "nada", "ninguém"}

def lematizar(texto):
    doc = nlp(texto)
    return " ".join(
        token.lemma_.lower() for token in doc
        if not token.is_punct and (not token.is_stop or token.lower_ in PALAVRAS_NEGACAO)
    )

corpus_lematizado = [lematizar(t) for t in textos]
print(corpus_lematizado[:3])
print(corpus_lematizado[50:53])

['financiamento rápido eficiente', 'corretor tranquilo início', 'atendimento bom recomendar']
['entrega chave decepção', 'corretor não resolver', 'apartamento decepção']


In [7]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    corpus_lematizado, rotulos,
    test_size=0.25,
    random_state=42,
    stratify=rotulos
)

print(f"Treino: {len(X_treino)} avaliações | Teste: {len(X_teste)} avaliações")

Treino: 75 avaliações | Teste: 25 avaliações


In [8]:
from collections import Counter
print(Counter(y_treino), Counter(y_teste))

Counter({'Positivo': 38, 'Negativo': 37}) Counter({'Negativo': 13, 'Positivo': 12})
